# AHP-Experiment: Auswertung

Mehrere Ergebnisdateien im Vergleich: LLM-Laeufe aus `experiment.py` und die
Probandendaten aus `transform_interviews.py`. Die Dateien stehen in `FILES`, die
erste ist die Referenz fuer die Abweichungstabelle.
Gewichte und CR rechnet pyDecision (`ahp_method`), das Notebook baut nur die Matrizen.

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
from pyDecision.algorithm import ahp_method        # pip install pyDecision

# Beliebig viele Ergebnisdateien; die erste ist die Referenz fuer den Vergleich.
FILES = [

    Path("../results/InterviewResults/data_ahpsupplierselectionjs23_2026-09-19_12-33_clean.json"),
    Path("../results/20260924-101539_openai_gpt-5.4-mini-2026-03-17_stateless.json"),
    Path("../results/20260924-101309_anthropic_claude-haiku-4-5-20251001_stateless.json"),
    Path("../results/20260924-102357_mistral_mistral-small-2603_stateless.json"),
]
FIGDIR = Path("figures"); FIGDIR.mkdir(exist_ok=True)

CRIT = ["liefertreue", "flexibilitaet", "kosten"]      # feste Reihenfolge = Matrixachsen
LABEL = {"liefertreue": "Liefertreue", "flexibilitaet": "Flexibilitaet", "kosten": "Kosten"}
CASE_LABEL = {"ketten": "Ketten", "batteriepacks": "Batteriepacks",
              "konnektivitaetsmodule": "Konnektivitaetsmodule"}
# Gewichtungsverfahren in ahp_method: "g" geometrisches Mittel, "me" Eigenvektor
# (bei 3x3-Matrizen identisch), "m" Mittel der normierten Spalten. CR jeweils
# nach Saaty mit dem Random Index von pyDecision (0,58 bei n = 3).
AHP_METHOD = "g"


def to_matrix(comparisons):
    """Drei Paarvergleiche -> reziproke 3x3-Matrix."""
    idx = {c: i for i, c in enumerate(CRIT)}
    A = np.ones((len(CRIT), len(CRIT)))
    for c in comparisons:
        i, j = idx[c["a"]], idx[c["b"]]
        s = max(int(c["intensity"]), 1)                 # ein Run liefert 0 statt 1
        if c["preference"] == "gleich":
            v = 1.0
        elif c["preference"] == c["a"]:
            v = float(s)
        elif c["preference"] == c["b"]:
            v = 1.0 / s
        else:
            raise ValueError(f"unbekannte preference: {c}")
        A[i, j], A[j, i] = v, 1.0 / v
    return A


def source_label(meta):
    """'Probanden' oder Modell mit Bedingung."""
    if meta["condition"] == "human":
        return "Probanden"
    return f'{meta["model"]} ({meta["condition"]})'


SOURCES, frames = [], []
for nr, path in enumerate(FILES, start=1):
    raw = json.loads(path.read_text(encoding="utf-8"))
    quelle = source_label(raw["meta"])
    if quelle in SOURCES:                               # gleiche Quelle mehrfach: Nummer anhaengen
        quelle = f"{quelle} #{nr}"
    SOURCES.append(quelle)
    print(f"{quelle}: {raw['summary']} | temp {raw['meta']['temperature']} | seed {raw['meta']['seed']}")

    records = []
    for r in raw["responses"]:
        # Ungueltige Antworten nicht auswerten: to_matrix wuerde fehlende
        # Vergleiche still als "gleich" behandeln.
        if not r["valid"]:
            continue
        w, cr = ahp_method(to_matrix(r["comparisons"]), wd=AHP_METHOD)
        records.append({
            "quelle": quelle, "run": r["run"], "case": r["case_id"], "position": r["position"],
            # stateless: jeder Fall einzeln abgefragt, es gibt keine Reihenfolge (None)
            "sequence": "-".join(s[:3] for s in r["sequence"]) if r["sequence"] else "stateless",
            **{c: w[i] for i, c in enumerate(CRIT)},
            "CR": cr,
        })
    frames.append(pd.DataFrame(records))

df = pd.concat(frames, ignore_index=True)
df["ranking"] = df[CRIT].apply(
    lambda r: " > ".join(LABEL[c] for c in r.sort_values(ascending=False).index), axis=1)
df["quelle"] = pd.Categorical(df["quelle"], categories=SOURCES, ordered=True)
df["case"] = pd.Categorical(df["case"], categories=list(CASE_LABEL), ordered=True)
df = df.sort_values(["quelle", "case", "run"]).reset_index(drop=True)
df.head()

## Kennzahlen

Je Fall und Quelle; `n` ist die Zahl der Runs bzw. Teilnahmen.

In [4]:
by = ["case", "quelle"]
n = df.groupby(by, observed=True).size().rename("n")
mean_w = df.groupby(by, observed=True)[CRIT].mean().rename(columns=LABEL).join(n)
cr_tab = df.groupby(by, observed=True)["CR"].agg(
    mean="mean", median="median", max="max",
    anteil_ueber_0_1=lambda s: (s > 0.1).mean())
patterns = df.groupby(by, observed=True).apply(
    lambda g: g[CRIT].round(6).apply(tuple, axis=1).nunique(), include_groups=False)

print("Mittlere Gewichte\n", mean_w.round(3), "\n")
print("Konsistenz\n", cr_tab.round(3), "\n")
print("Verschiedene Antwortmuster je Fall\n", patterns.to_string())

Mittlere Gewichte
                                                            Liefertreue  \
case                  quelle                                             
ketten                Probanden                                  0.516   
                      gpt-4o-mini-2024-07-18 (history)           0.630   
                      claude-haiku-4-5-20251001 (history)        0.620   
                      mistral-small-2603 (history)               0.559   
batteriepacks         Probanden                                  0.239   
                      gpt-4o-mini-2024-07-18 (history)           0.153   
                      claude-haiku-4-5-20251001 (history)        0.103   
                      mistral-small-2603 (history)               0.171   
konnektivitaetsmodule Probanden                                  0.314   
                      gpt-4o-mini-2024-07-18 (history)           0.160   
                      claude-haiku-4-5-20251001 (history)        0.186   
                   

## Vergleich mit der Referenz

Abweichung der mittleren Gewichte von der ersten Datei in `FILES` (positiv =
hoeher gewichtet als die Referenz) und die haeufigste Rangfolge je Quelle.

In [5]:
REF = SOURCES[0]
mean_all = df.groupby(["case", "quelle"], observed=True)[CRIT].mean()
diff = (mean_all.sub(mean_all.xs(REF, level="quelle"), level="case")
                .drop(REF, level="quelle").rename(columns=LABEL))
print(f"Abweichung der mittleren Gewichte von: {REF}\n", diff.round(3), "\n")


def top_ranking(s):
    anteil = s.value_counts(normalize=True)
    return f"{anteil.index[0]} ({anteil.iloc[0]:.0%})"


print("Haeufigste Rangfolge (Anteil)")
df.groupby(["case", "quelle"], observed=True)["ranking"].agg(top_ranking).unstack("quelle")

Abweichung der mittleren Gewichte von: Probanden
                                                            Liefertreue  \
case                  quelle                                             
ketten                gpt-4o-mini-2024-07-18 (history)           0.115   
                      claude-haiku-4-5-20251001 (history)        0.104   
                      mistral-small-2603 (history)               0.043   
batteriepacks         gpt-4o-mini-2024-07-18 (history)          -0.086   
                      claude-haiku-4-5-20251001 (history)       -0.136   
                      mistral-small-2603 (history)              -0.068   
konnektivitaetsmodule gpt-4o-mini-2024-07-18 (history)          -0.154   
                      claude-haiku-4-5-20251001 (history)       -0.128   
                      mistral-small-2603 (history)              -0.152   

                                                           Flexibilitaet  \
case                  quelle                               

quelle,Probanden,gpt-4o-mini-2024-07-18 (history),claude-haiku-4-5-20251001 (history),mistral-small-2603 (history)
case,,,,
ketten,Liefertreue > Kosten > Flexibilitaet (65%),Liefertreue > Kosten > Flexibilitaet (100%),Liefertreue > Kosten > Flexibilitaet (100%),Liefertreue > Kosten > Flexibilitaet (70%)
batteriepacks,Flexibilitaet > Kosten > Liefertreue (40%),Flexibilitaet > Kosten > Liefertreue (76%),Flexibilitaet > Kosten > Liefertreue (98%),Flexibilitaet > Liefertreue > Kosten (94%)
konnektivitaetsmodule,Flexibilitaet > Liefertreue > Kosten (40%),Flexibilitaet > Liefertreue > Kosten (76%),Flexibilitaet > Liefertreue > Kosten (100%),Flexibilitaet > Liefertreue > Kosten (100%)


## Abbildungsstil

Einmal global, danach nicht mehr anfassen. Vektorexport. Jede Quelle hat eine Farbe
und zusaetzlich einen Marker bzw. eine Schraffur, damit die Zuordnung auch in
Graustufen und bei Farbfehlsichtigkeit lesbar bleibt.

In [11]:
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from matplotlib.patches import Patch

mpl.rcParams.update({
    "figure.dpi": 110,
    "savefig.dpi": 300,
    "savefig.bbox": "tight",
    "savefig.transparent": False,
    "font.family": "serif",
    "font.size": 9,
    "axes.titlesize": 9,
    "axes.labelsize": 9,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "grid.linewidth": 0.4,
    "grid.alpha": 0.4,
    "legend.frameon": False,
    "lines.markersize": 4,
    "hatch.linewidth": 0.8,
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
})

# Feste Reihenfolge je Quelle, nie zyklisch neu vergeben. Die ersten drei Farben
# sind auch bei Farbfehlsichtigkeit paarweise unterscheidbar; ab der vierten
# Quelle traegt vor allem Marker bzw. Schraffur die Zuordnung.
COLORS = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#008300", "#4a3aa7", "#e34948"]
MARKERS = ["o", "s", "^", "D", "v", "P", "X", "*"]
HATCHES = ["", "////", "\\\\\\\\", "xxxx", "....", "----", "||||", "++++"]
if len(SOURCES) > len(COLORS):
    raise ValueError(f"Hoechstens {len(COLORS)} Dateien in FILES")
STYLE = {q: {"color": COLORS[i], "marker": MARKERS[i], "hatch": HATCHES[i]}
         for i, q in enumerate(SOURCES)}
K = len(SOURCES)
SLOT = 0.8                                  # Breite, die sich die Quellen je Kategorie teilen
OFFSET = {q: (i - (K - 1) / 2) * SLOT / K for i, q in enumerate(SOURCES)}
HALF = SLOT / K / 2                         # halbe Breite je Quelle


def legend(fig, bars=False):
    """Legende ueber der Abbildung; Marker fuer Punkte, Schraffur fuer Balken."""
    if bars:
        handles = [Patch(facecolor=STYLE[q]["color"], hatch=STYLE[q]["hatch"],
                         edgecolor="white", linewidth=0, label=q) for q in SOURCES]
    else:
        handles = [Line2D([], [], ls="", markersize=5, color=STYLE[q]["color"],
                          marker=STYLE[q]["marker"], label=q) for q in SOURCES]
    fig.legend(handles=handles, loc="lower center", bbox_to_anchor=(0.5, 1.0),
               ncol=min(K, 3))


def save(fig, name):
    for ext in ("pdf", "svg"):
        fig.savefig(FIGDIR / f"{name}.{ext}")
    print("gespeichert:", name)

## Abbildung 1 — Streuung der Gewichte

Jeder Punkt ist ein Run bzw. eine Teilnahme, der Strich der Mittelwert je Quelle.

In [16]:
rng = np.random.default_rng(0)
fig, axes = plt.subplots(1, 3, figsize=(7.0, 2.6), sharey=True)

for ax, (case, g) in zip(axes, df.groupby("case", observed=True)):
    for q, gq in g.groupby("quelle", observed=True):
        for i, c in enumerate(CRIT):
            x, y = i + OFFSET[q], gq[c].to_numpy()
            ax.scatter(x + rng.uniform(-0.6, 0.6, y.size) * HALF, y, s=14, alpha=0.45,
                       color=STYLE[q]["color"], marker=STYLE[q]["marker"],
                       edgecolors="none", zorder=3)
            ax.hlines(y.mean(), x - HALF, x + HALF, color="black", lw=1.4, zorder=4)
    ax.set_xticks(range(len(CRIT)))
    ax.set_xticklabels([LABEL[c] for c in CRIT], rotation=30, ha="right")
    ax.set_title(CASE_LABEL[case])
    ax.set_xlim(-0.5, len(CRIT) - 0.5)
    ax.xaxis.grid(False)

axes[0].set_ylim(0, 1)
axes[0].set_ylabel("Gewicht")
legend(fig)
save(fig, "01_gewichte")
plt.show()

gespeichert: 01_gewichte


## Abbildung 2 — Konsistenz

Logarithmische Achse, sonst erdrueckt ein einzelner hoher CR alles andere. Perfekt
konsistente Urteile (CR = 0) liegen am unteren Rand, weil 0 auf der log-Achse nicht
darstellbar ist. Die Zahl ueber jeder Spalte ist der Anteil mit CR > 0,1.

In [13]:
CR_FLOOR = 6e-3                              # CR = 0 -> unterer Rand
fig, ax = plt.subplots(figsize=(4.6, 2.8))

for i, case in enumerate(CASE_LABEL):
    for q in SOURCES:
        y = df.loc[(df["case"] == case) & (df["quelle"] == q), "CR"].to_numpy()
        if y.size == 0:
            continue
        x = i + OFFSET[q]
        ax.scatter(x + rng.uniform(-0.6, 0.6, y.size) * HALF, np.clip(y, CR_FLOOR, None),
                   s=14, alpha=0.5, color=STYLE[q]["color"], marker=STYLE[q]["marker"],
                   edgecolors="none", zorder=3)
        ax.annotate(f"{(y > 0.1).mean():.0%}", (x, 2.2), ha="center", fontsize=7)

ax.annotate("CR > 0,1:", (-0.5, 2.2), xytext=(-4, 0), textcoords="offset points",
            ha="right", fontsize=7)
ax.axhline(0.1, color="black", lw=0.9, ls="--", zorder=2)
ax.set_yscale("log")
ax.set_ylim(5e-3, 4)
ax.set_xlim(-0.5, len(CASE_LABEL) - 0.5)
ax.set_xticks(range(len(CASE_LABEL)))
ax.set_xticklabels([CASE_LABEL[c] for c in CASE_LABEL], rotation=20, ha="right")
ax.set_ylabel("Consistency Ratio")
ax.xaxis.grid(False)

sec = ax.secondary_yaxis("right")
sec.set_yticks([0.1]); sec.set_yticklabels(["0,1"])
sec.tick_params(length=0)
sec.spines["right"].set_visible(False)
sec.minorticks_off()

legend(fig)
save(fig, "02_konsistenz")
plt.show()

gespeichert: 02_konsistenz


## Abbildung 3 — Rangfolgen

Anteil je Quelle statt Anzahl, weil die Quellen unterschiedlich viele Runs bzw.
Teilnahmen haben. Sortiert nach mittlerem Anteil ueber alle Quellen.

In [14]:
share = (df.groupby(["case", "quelle"], observed=True)["ranking"]
           .value_counts(normalize=True).rename("anteil").reset_index())

groups = []
for case in CASE_LABEL:
    g = (share[share["case"] == case]
         .pivot(index="ranking", columns="quelle", values="anteil")
         .reindex(columns=SOURCES).fillna(0))
    groups.append((case, g.loc[g.mean(axis=1).sort_values().index]))

rows = sum(len(g) for _, g in groups)
height = 0.9 + 0.16 * K * rows
fig, axes = plt.subplots(len(groups), 1, figsize=(5.6, height), sharex=True,
                         gridspec_kw={"height_ratios": [len(g) for _, g in groups],
                                      "hspace": 0.75, "top": 1 - 0.3 / height})

for ax, (case, g) in zip(axes, groups):
    for q in SOURCES:
        ypos = np.arange(len(g)) - OFFSET[q]           # erste Quelle oben
        ax.barh(ypos, g[q], height=SLOT / K * 0.9, color=STYLE[q]["color"],
                hatch=STYLE[q]["hatch"], edgecolor="white", linewidth=0)
        for y, v in zip(ypos, g[q]):
            if v > 0:
                ax.annotate(f"{v:.0%}", (v + 0.01, y), va="center", fontsize=7)
    ax.set_yticks(range(len(g))); ax.set_yticklabels(g.index)
    ax.set_ylim(-0.6, len(g) - 0.4)
    ax.yaxis.grid(False)
    ax.set_title(CASE_LABEL[case], loc="left", style="italic", pad=3)

axes[-1].set_xlim(0, 1.1)
axes[-1].xaxis.set_major_formatter(mpl.ticker.PercentFormatter(1.0))
axes[-1].set_xlabel("Anteil der Runs bzw. Teilnahmen je Quelle")
legend(fig, bars=True)
save(fig, "03_rangfolgen")
plt.show()

gespeichert: 03_rangfolgen


## Export

Ein Long-Format fuer alles Weitere, eine Zeile je Quelle, Run und Fall. Semikolon
und Dezimalkomma, damit Excel nicht meckert.

In [15]:
out = df[["quelle", "run", "case", "position", "sequence", *CRIT, "CR", "ranking"]]
out.to_csv("runs.csv", sep=";", decimal=",", encoding="utf-8-sig", index=False)
print(f"{len(out)} Zeilen -> runs.csv")
out.tail(3)

510 Zeilen -> runs.csv


,quelle,run,case,position,sequence,liefertreue,flexibilitaet,kosten,CR,ranking
507,mistral-small-2603 (history),48,konnektivitaetsmodule,0,kon-bat-ket,0.192880,0.700974,0.106146,0.221268,Flexibilitaet > Liefertreue > Kosten
508,mistral-small-2603 (history),49,konnektivitaetsmodule,0,kon-ket-bat,0.186705,0.742912,0.070383,0.147475,Flexibilitaet > Liefertreue > Kosten
509,mistral-small-2603 (history),50,konnektivitaetsmodule,0,kon-ket-bat,0.184307,0.721536,0.094157,0.159566,Flexibilitaet > Liefertreue > Kosten
